# Day 12 — Project Structure, pyproject.toml & uv, Logging, ruff & mypy (5 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: turn a one-file script into a real Python project that anyone can install with one command, that reports what it is doing through logs instead of print(), and that two automatic reviewers (ruff for style and likely bugs, mypy for types) pass before any human reads it.

- Deliverable: day12/cms-export/, the customer's legacy export script rebuilt as the installable package cms_export with a cms-export command, logging controlled by -v and -q, the 11 given tests plus at least 3 of your own green, ruff and mypy --strict clean, and a wheel that installs into a fresh venv. Plus day12/dsa.py.

- Where this sits: Week 2, Phase 1 (Foundation), day 5 of 7. Every project from here on (the FastAPI service in Week 4, Project 1 in Week 8, the CMS agent in Week 11) starts with uv init --package and today's checklist, so this is the shape your portfolio repos will have.

Time split (5h):

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 6, typing every example in a scratch folder |
| Hands-on | 2.5h | Parts A to D: project, port, logging, quality gates |
| DSA | 0.5h | Compare versions, common package prefix |
| Revision | 0.5h | Quiz, Remember cells in the notebook, a clean commit |

Open items first: GitHub still shows only day01/ to day04/, with Day 4's starter TODOs. Days 5 to 11 are not pushed. If that work is on your laptop, push it before you start (one branch per day is fine). Today's task is self-contained (it ships its own legacy script and sample data), so you can do it even if earlier days are unfinished. The Day 14 score reads the repo.

Install today: uv. On Mac or Linux run curl -LsSf https://astral.sh/uv/install.sh | sh; on Windows PowerShell run powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex". Then uv --version should print 0.11 or newer. Versions used for this session: Python 3.13, uv 0.11.32, ruff 0.17.3, mypy 2.4.1, pytest 9.2.0.

## Recap (2 minutes, from memory)

- A function has yield in its body. What does calling it return, and when does the code inside actually run?

- Why does @functools.wraps(func) matter in a decorator? Name one thing that breaks without it.

- Your atomic_csv context manager writes to a .tmp file and then calls os.replace(). If the export crashes halfway, what happens to an existing out.csv, and why?

Answers are on the last page of today's session.

## Concept 1: Modules, packages and the src layout

### Why it matters for an FDE

In Week 1 every task was one file that you ran from its own folder. Customer code you hand over is never used like that. It has to be installed on a job runner, in a Docker image (Week 10) or on a colleague's laptop, and run by people who never read it. If it only works when you first cd into the right folder, it does not work.

The first thing a senior FDE does with a customer's script is give it a shape: a package with one job per module, one entry point and an install step. In a code review, that is what "production-ready" means before anyone talks about scale.

### How it works

A module is one .py file. Importing it runs the whole file once, top to bottom, and Python caches the result, so a second import does nothing. Everything at the top level (constants, function definitions, and any stray print()) runs at import time. That is why the legacy script you get today is unsafe to import: its last line calls main().

In [ ]:
# Day 12: what "import" really does. Save as pricing.py, then run the
# two commands at the bottom from the same folder.
"""Tiny module used to show how imports and __name__ work."""

GST_RATE = 0.18  # top-level code runs once, at the first import

print("pricing loaded, __name__ =", __name__)  # runs on import too


def with_gst(amount: float) -> float:
    """Return the amount including 18% GST."""
    return round(amount * (1 + GST_RATE), 2)


# True only when the file is run directly: like require.main === module
if __name__ == "__main__":
    print(with_gst(100))

# $ python3 pricing.py
# -> pricing loaded, __name__ = __main__
# -> 118.0
# $ python3 -c "import pricing; import pricing; print(pricing.with_gst(50))"
# -> pricing loaded, __name__ = pricing   (once: imports are cached)
# -> 59.0

__name__ is the module's own name while it is imported, and the string "__main__" when you run the file directly. The if __name__ == "__main__": guard is Python's version of require.main === module in Node: the code under it runs for python3 pricing.py, never on import.

A package is a folder of modules with an __init__.py, like a folder with an index.js. You import from it with dotted names: from cms_export.reader import read_export. Inside a package, use these absolute imports everywhere; they read the same in every file and every tool understands them. A __main__.py inside the package lets people run python -m cms_export, the way node . runs the entry file named in package.json.

The src layout puts the package inside a src/ folder, next to the tests and the project files rather than among them. Today's target looks like this:

```bash
# day12/cms-export/ after today: the "src layout"
cms-export/                 # project folder: dashes are fine here
    pyproject.toml          # project metadata, like package.json
    uv.lock                 # exact versions, like package-lock.json
    .venv/                  # this project's venv (never committed)
    src/
        cms_export/         # the import package: underscore, no dash
            __init__.py     # makes the folder a package (like index.js)
            __main__.py     # makes "python -m cms_export" work
            cli.py          # argv, exit codes, logging setup
            reader.py       # file in, entries out
            transform.py    # pure rules: entry in, row out
            writer.py       # rows in, CSV out
            py.typed        # tells mypy "this package has type hints"
    tests/
        test_transform.py   # imports cms_export like a customer would
        test_cli.py
```

Why the extra folder? Python puts the current folder on the import path. In a flat layout your tests import the package straight from the folder, so they pass even when the installed package is broken (a file left out of the build, a wrong entry point). With src/ the package cannot be imported until it is installed, so your tests check what the customer will actually get.

```bash
# Day 12: src/ makes "works on my machine" fail early, on purpose
$ uv init --package ticket-report   # a new project with the src layout
$ cd ticket-report
$ python3 -c "import ticket_report"
# -> ModuleNotFoundError: No module named 'ticket_report'
# Good: src/ is not on the import path, so nothing imports by accident.
$ uv sync                           # creates .venv, installs the project
# -> + ticket-report==0.1.0 (from file:///.../ticket-report)
$ uv run python -c "import ticket_report; print(ticket_report.__file__)"
# -> /.../ticket-report/src/ticket_report/__init__.py
# An editable install: the venv points at src/, so edits apply at once.
```

Editable install: uv sync installs your own project in editable mode, which means the venv points at src/ instead of copying it. You edit a file and the next run sees the change, with no reinstall. It is the same idea as npm link.

### Real-world FDE example

A retailer's data team hands you sync_products.py: 900 lines that push their product catalogue into Contentstack every night, from a cron job on one VM, with a hard-coded path. Nobody dares touch it. You split it into a package: catalog_sync.source reads their ERP export, catalog_sync.mapping holds the pure rules (ERP fields to Contentstack fields, fully tested), catalog_sync.cms makes the API calls with retries, and catalog_sync.cli handles arguments and logging. The mapping rules change every month; now each change is a small, tested diff in one module instead of a risky edit in the middle of a script.

### Common mistakes

- Naming a module after a standard library or installed package. A file called logging.py, json.py or requests.py in your folder hides the real one, and the error appears somewhere else ("module 'logging' has no attribute 'getLogger'").

- Using a dash in the import name. The project and the command can be called cms-export, but the package folder must be cms_export: import cms-export is a syntax error.

- Doing real work at import time: reading files, calling APIs or configuring logging at the top level of a module. Every import, including the ones pytest does, runs it.

- Circular imports: reader imports cli and cli imports reader. Keep one direction: cli at the top, pure modules such as transform at the bottom, importing nothing of yours.

- Leaving out __init__.py. Python can import some folders without one (namespace packages), but build backends and mypy treat them differently. Always add it.

## Concept 2: pyproject.toml, one file that describes the project

### Why it matters for an FDE

When you join a customer project, pyproject.toml is the first file to open. It tells you the Python version, what the code depends on, which commands it installs, and how linting, typing and tests are configured. When you hand a project over, it is the contract: their CI and their platform team read it, not your README.

It replaced a pile of older files (setup.py, setup.cfg, requirements.txt, tox.ini, .flake8) that you will still meet in older customer repos. Knowing what each table means lets you modernise those repos safely.

### How it works

pyproject.toml is written in TOML, a small config format: [table] headers, key = value lines and # comments. You already know its ideas from package.json. Here is a complete one for a small tool, annotated:

```bash
# pyproject.toml for a small tool (in TOML, comments also start with #)
[project]
name = "ticket-report"         # the install name, like "name" in npm
version = "0.1.0"              # the ONE place the version lives
description = "Summarise support tickets from a CSV"
readme = "README.md"
requires-python = ">=3.12"     # like "engines" in package.json
dependencies = [               # runtime needs, like "dependencies"
    "requests>=2.34.2",        # a lower bound, not an exact pin
]

[project.scripts]              # like "bin" in package.json
# command name = "package.module:function"
ticket-report = "ticket_report.cli:main"

[dependency-groups]            # dev-only tools, like "devDependencies"
dev = ["mypy>=2.4.1", "pytest>=9.2.0", "ruff>=0.17.3"]

[build-system]                 # the tool that turns src/ into a wheel
requires = ["uv_build>=0.11.32,<0.12.0"]
build-backend = "uv_build"

[tool.ruff]                    # each tool reads only its own [tool.x]
line-length = 88
```

The tables you will touch most:

- [project]: the standard metadata every Python tool reads. dependencies lists what the code needs at runtime, with a lower bound such as >=2.34.2, not an exact pin; exact versions live in the lock file (Concept 3).

- [project.scripts]: each line creates a command in the venv when the project is installed. cms-export = "cms_export.cli:main" means "run main() from the module cms_export.cli". This is how uv, ruff and pytest themselves become commands.

- [dependency-groups]: tools you need while developing (pytest, ruff, mypy) that the customer does not need in production, like devDependencies.

- [build-system]: the build backend that turns your source into an installable file. uv init --package picks uv's own backend, uv_build; you will also meet hatchling and setuptools. You rarely change this table.

- [tool.<name>]: settings for one tool, read by that tool only: [tool.ruff], [tool.mypy], [tool.pytest.ini_options].

Wheel: building a project produces a wheel (a .whl file, which is a zip of your package plus metadata) and an sdist (a .tar.gz of the source). pip and uv install wheels. A wheel is what you upload to a customer's private package index or copy into a Docker image.

The version in [project] can be read at runtime with importlib.metadata.version("cms-export"). Use that for a --version flag instead of typing the number a second time. It reads the installed metadata, so it only works once the project is installed, which is one more reason for the src layout.

### Real-world FDE example

A bank's platform team says your agent service must install from their internal package mirror, must run on Python 3.12 (their approved base image), and must not pull in anything their security team has not reviewed. All three answers live in pyproject.toml: requires-python = ">=3.12", a short dependencies list the security team can check line by line, and dev tools kept in [dependency-groups] so they never reach production. You send them this file before you send any code.

### Common mistakes

- Putting pytest or ruff in dependencies: every production install now pulls in your test tools.

- Exact pins in dependencies (requests==2.34.2) in a package other code will import. Two packages pinning different exact versions cannot be installed together. Pin exactly in the lock file, not in the metadata.

- An entry point that points at the wrong place. uv init --package writes cms-export = "cms_export:main", which means main() in __init__.py. When main() lives in cli.py, change the line, or the command fails with an ImportError.

- Keeping the version number in two places. Keep it in pyproject.toml only and read it with importlib.metadata.

## Concept 3: uv, one tool for Python, venvs, dependencies and builds

### Why it matters for an FDE

On Day 6 you did it the classic way: create a venv, activate it, pip install, then pip freeze into requirements.txt. That works, but every step can be forgotten, and requirements.txt cannot tell your direct dependencies from theirs. uv (from Astral, the makers of ruff) does all of it in one fast tool, and it is now the default in most new Python projects, including many AI SDK examples.

Customers will ask how to reproduce your environment exactly. With uv the answer is two files in Git (pyproject.toml and uv.lock) and one command.

### How it works

The everyday loop is short. Every command below was run for this session; long outputs are trimmed.

```bash
# Day 12: the uv commands you will use every day (real output, trimmed)
$ uv init --package ticket-report   # new project with the src/ layout
# -> Initialized project `ticket-report`
$ cd ticket-report
$ uv add requests                   # edits pyproject, locks, installs
# -> Creating virtual environment at: .venv
# -> + requests==2.34.2             (plus its 4 dependencies)
$ uv add --dev pytest               # goes into [dependency-groups] dev
$ uv tree --depth 1                 # what depends on what
# -> ticket-report v0.1.0
# -> ├── requests v2.34.2
# -> └── pytest v9.2.0 (group: dev)
$ uv run ticket-report              # runs inside .venv: no "activate"
# -> Hello from ticket-report!
$ uv sync --locked                  # install EXACTLY what uv.lock says
$ uv lock --upgrade-package requests   # move one package, on purpose
$ uv build                          # sdist and wheel into dist/
# -> Successfully built dist/ticket_report-0.1.0.tar.gz
# -> Successfully built dist/ticket_report-0.1.0-py3-none-any.whl
$ uvx ruff --version                # run a tool once, like npx
# -> ruff 0.17.3
```

What each piece does:

- uv add edits pyproject.toml, resolves versions, updates uv.lock and installs into .venv in one step. uv remove undoes it.

- uv.lock records the exact version and hash of every package, including the dependencies of your dependencies. Commit it. Anyone who runs uv sync gets exactly the same set on any OS, like npm ci with package-lock.json.

- uv run <command> first makes sure .venv matches the lock file, then runs the command inside it. You never activate the venv, which removes the most common Day 6 mistake: installing into the wrong Python.

- uv sync --locked fails instead of quietly updating the lock when pyproject.toml and uv.lock disagree. Use it in CI and in Dockerfiles so a build never changes versions on its own.

- uvx <tool> (short for uv tool run) runs a command-line tool in a throwaway environment, like npx.

- uv can install Python itself: uv python install 3.12. The .python-version file written by uv init pins the version the project uses.

| You want to | npm | uv |
|---|---|---|
| Start a project | npm init | uv init --package |
| Add a dependency | npm install axios | uv add requests |
| Add a dev-only tool | npm install -D jest | uv add --dev pytest |
| Install exactly from the lock | npm ci | uv sync --locked |
| Run a project command | npm run / npx | uv run |
| Run a tool once | npx prettier | uvx ruff |
| Lock file | package-lock.json | uv.lock |

FDE TIP: customers with a locked-down network often allow packages only from an internal mirror. uv supports that with --index-url or an [[tool.uv.index]] table in pyproject.toml. Ask about the package mirror in the first technical call, not on deployment day.

### Real-world FDE example

A customer's CI fails on Monday morning: the RAG service you shipped on Friday now crashes on import. Nothing in your code changed. A dependency released a new version over the weekend, and their build ran pip install -r requirements.txt with loose versions. With uv.lock committed and the Dockerfile running uv sync --locked, Monday's build would have installed Friday's exact versions. Upgrades then become a deliberate change: uv lock --upgrade-package <name>, run the tests, open a pull request.

### Common mistakes

- Not committing uv.lock, or committing .venv/. The lock file belongs in Git; the venv never does, because it is rebuilt from the lock in seconds.

- Running pip install inside a uv project. The package lands in the venv but not in pyproject.toml or the lock, so it disappears on the next uv sync. Use uv add.

- Running uv init --package in a folder that is not inside a Git repo: uv then creates a new .git folder there. Inside your fde-journey repo it does not. If you ever get a nested .git folder, delete only that inner folder.

- Editing uv.lock by hand. It is generated; change pyproject.toml or use uv commands instead.

## Concept 4: Logging, messages for the people who run your code

### Why it matters for an FDE

print() talks to whoever is watching the terminal. In production nobody is watching. Your code runs in a customer's job runner, a container or a serverless function, and the only record of what happened is the log stream their platform collects (CloudWatch, Datadog, Splunk). When something breaks at 3 a.m., the on-call engineer filters those logs by level and by module. Unstructured prints on stdout, mixed in with the CSV data, cannot be filtered.

Logging is also how you debug a deployment you cannot attach a debugger to. You ship with INFO, and when the customer reports a problem you ask them to rerun with -v and send you the DEBUG lines.

### How it works

Three ideas carry most of the weight. A logger is a named object your module writes to; create one per module with logging.getLogger(__name__). A level says how serious a message is: DEBUG, INFO, WARNING, ERROR and CRITICAL, in that order (they are the numbers 10 to 50). Configuration, meaning where messages go, which levels show and what each line looks like, happens once, in the program's entry point.

In [ ]:
# Day 12: logging basics and where output goes (save as logdemo.py)
import logging
import sys

# one logger per module, named after it; in a real module, use __name__
logger = logging.getLogger("cms_export.reader")

# configure ONCE, in the program's entry point (main), never in a library
logging.basicConfig(
    level=logging.INFO,  # show INFO and above; DEBUG is dropped
    format="%(levelname)s %(name)s: %(message)s",
    stream=sys.stderr,  # messages on stderr, data on stdout (Day 9)
)

logger.debug("opened export file")  # below INFO: dropped, almost free
logger.info("read %d entries", 7)  # -> INFO cms_export.reader: read 7 entries
logger.warning("skipping line %d", 6)
# -> WARNING cms_export.reader: skipping line 6
logger.error("export file not found: %s", "nope.jsonl")
# -> ERROR cms_export.reader: export file not found: nope.jsonl
print("uid,title")  # -> uid,title   (stdout: the data channel)

# $ python3 logdemo.py 2>/dev/null    shows only:  uid,title
# $ python3 logdemo.py 1>/dev/null    shows only the three log lines

Logger names form a tree, split on the dots: cms_export.reader is a child of cms_export. A record travels up the tree to the root logger, whose handler writes it out. That is why __name__ is the right name: it puts every module's logger under the package name automatically, and lets you turn one part of the system up or down without touching the rest.

In [ ]:
# Day 12: logger names form a tree; lazy formatting; logging exceptions
import json
import logging

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s"
)

reader_log = logging.getLogger("cms_export.reader")  # child of cms_export
cli_log = logging.getLogger("cms_export.cli")

# turn ONE part of the app down without touching the others
logging.getLogger("cms_export.reader").setLevel(logging.WARNING)
reader_log.info("hidden: the reader logger now needs WARNING")
cli_log.info("still shown")  # -> INFO cms_export.cli: still shown

# noisy third-party libraries are quietened the same way
logging.getLogger("urllib3").setLevel(logging.WARNING)

# FDE TIP: pass values as arguments, not f-strings; the text is built
# only if the message is actually shown
cli_log.debug("entry %s", {"uid": "blt001"})  # never formatted: DEBUG off

try:
    json.loads('{"uid": "blt005"')  # a broken line from the export
except json.JSONDecodeError:
    # .exception() = ERROR level + the full traceback; use it in except
    cli_log.exception("could not parse line %d", 6)
# -> ERROR cms_export.cli: could not parse line 6
# -> Traceback (most recent call last):
# ->   ... (the frames, as in any traceback)
# -> json.decoder.JSONDecodeError: Expecting ',' delimiter: line 1 ...

The rules that make logs useful:

- Choose the level by who has to act. DEBUG: detail for you while diagnosing. INFO: normal milestones ("wrote 5 rows"). WARNING: something odd that the run survived, such as a skipped bad line. ERROR: this operation failed. CRITICAL: the program cannot continue.

- Pass values as arguments: logger.info("read %d entries", n), not an f-string. The text is only built if the message is shown, and log platforms can group messages that share a template.

- Inside except, use logger.exception(...): it logs at ERROR and adds the traceback.

- Libraries create loggers but never configure logging. Only main() calls basicConfig, so whoever imports your code (a FastAPI app, a test) keeps control of their own output.

- Never log secrets or personal data: API keys, tokens, full request headers, customer emails. Log a record's uid, not its contents.

Testing logs: pytest's built-in caplog fixture captures records during a test, so a warning becomes part of the contract, proven by a test:

In [ ]:
# Day 12: testing log output with pytest's built-in caplog fixture
# --- locale_fix.py (the module under test) ---
import logging

logger = logging.getLogger(__name__)


def pick_locale(entry: dict[str, str], default: str = "en-us") -> str:
    """Return the entry's locale, or the default with a warning."""
    if not entry.get("locale"):
        logger.warning("%s has no locale, using %s", entry["uid"], default)
        return default
    return entry["locale"]


# --- test_locale_fix.py ---
import logging

import pytest

from locale_fix import pick_locale


def test_warns_when_locale_missing(caplog: pytest.LogCaptureFixture) -> None:
    # caplog collects every log record made during the test
    with caplog.at_level(logging.WARNING):  # make sure WARNING is shown
        assert pick_locale({"uid": "blt006"}) == "en-us"
    assert "blt006 has no locale" in caplog.text  # the formatted text
    assert caplog.records[0].levelname == "WARNING"  # the record itself


def test_silent_when_locale_present(caplog: pytest.LogCaptureFixture) -> None:
    assert pick_locale({"uid": "blt001", "locale": "en-in"}) == "en-in"
    assert caplog.records == []  # REMEMBER: prove the noise is NOT there
# $ python3 -m pytest -q test_locale_fix.py
# -> 2 passed

GOTCHA: basicConfig only works the first time. If the root logger already has a handler, it silently does nothing, including the level you passed. Under pytest the root logger always has handlers (caplog's), which is why today's configure_logging sets the level with a separate logging.getLogger().setLevel(...) line. Do not fix it with basicConfig(force=True): force removes every existing handler, caplog's included, and your log tests go silent.

### Real-world FDE example

A media customer runs your nightly Contentstack sync on AWS ECS, and their editors say "some articles did not update". Because the sync logs one WARNING per skipped entry, with its uid and the reason, in the same format on every run, their engineer runs one CloudWatch query (level WARNING, logger content_sync.mapping, last 7 days) and finds 14 entries skipped for a missing locale, all created by one new regional team. The fix is a content rule, not a code change, and nobody had to read your code to find it.

### Common mistakes

- Calling basicConfig at the top of a library module. Whichever module is imported first wins, and the application loses control of its logging.

- Using the root logger directly (logging.info(...)) everywhere. Every line then says "root", and nobody can filter by module.

- Logging and re-raising the same error at every layer, so one failure appears five times. Log it once, where you handle it.

- Writing logs to stdout in a command whose stdout is data. Day 9's rule holds: data on stdout, messages on stderr, so a pipe such as cms-export ... | other-tool still works.

- Leaving chatty detail at INFO. If every run prints 10,000 INFO lines, nobody sees the one WARNING that matters.

## Concept 5: ruff, linting and formatting in one fast tool

### Why it matters for an FDE

A linter reads your code without running it and flags likely bugs and risky patterns; a formatter rewrites the layout so every file looks the same. You know both from ESLint and Prettier. ruff does both jobs for Python in milliseconds, and replaces flake8, isort, pyupgrade and black.

Customer engineering teams increasingly run ruff in CI, and a pull request that fails lint does not get reviewed. It also catches several of the bugs you have already met in this course, before you run anything.

### How it works

You choose rule families by code in pyproject.toml. A good starting set: E and W (style errors), F (pyflakes: unused imports, undefined names), I (import sorting), B (bugbear: likely bugs), UP (pyupgrade: modern syntax for your Python version) and SIM (simplifications). Here it is on a file with five classic problems:

```sql
# Day 12: messy.py, five classic problems in 17 lines
import os, json
from typing import List


def load(path, cache={}):
    if path in cache:
        return cache[path]
    try:
        data = json.load(open(path))
    except:
        data = None
    cache[path] = data
    return data


def titles(entries: List[dict]) -> List[str]:
    return [e["title"] for e in entries if e["status"] == "published"]

# pyproject.toml, next to it
[tool.ruff.lint]
select = ["E", "F", "W", "I", "B", "UP", "SIM"]

$ ruff check --output-format concise messy.py
# -> messy.py:1:1: E401 [*] Multiple imports on one line
# -> messy.py:1:1: I001 [*] Import block is un-sorted or un-formatted
# -> messy.py:1:8: F401 [*] `os` imported but unused
# -> messy.py:2:1: UP035 `typing.List` is deprecated, use `list` instead
# -> messy.py:5:22: B006 Do not use mutable data structures for ...
# -> messy.py:9:26: SIM115 Use a context manager for opening files
# -> messy.py:10:5: E722 Do not use bare `except`
# -> messy.py:16:21: UP006 [*] Use `list` instead of `List` for ...
# -> messy.py:16:36: UP006 [*] Use `list` instead of `List` for ...
# -> Found 9 errors.
$ ruff check --fix messy.py   # applies the 5 safe fixes, 3 remain
$ ruff format messy.py        # rewrites the layout, like Prettier
```

Read each line as file:line:column, rule code, message. [*] means ruff can fix it safely with --fix. The three it leaves alone need your judgement: the mutable default (Day 2's trap, rule B006), opening a file without with (Day 4, SIM115) and the bare except (Day 4, E722). Look up any rule with ruff rule B006.

The daily pair is ruff check --fix and ruff format. Run both before every commit. When a rule is wrong for one specific line, silence that line only and say why: # noqa: E501 with a short reason. Never switch off a whole rule family to make one warning go away.

### Real-world FDE example

You inherit a customer's 40-file integration service with no linter, and ruff check reports 1,300 issues. You do not fix them in one pull request. First, ruff check --fix and ruff format in a commit of their own, with no behaviour change, which is easy to review. Then you fix the real bugs ruff found (in a codebase like this, typically a few bare excepts that swallow API errors and a mutable default or two), each with a test. Finally, you add ruff to CI so the count never grows again. That order is how you improve a customer's codebase without alarming their team.

### Common mistakes

- Mixing a format-only change with logic changes in one commit. The reviewer cannot find the real change among 2,000 reformatted lines.

- Running --unsafe-fixes without reading the diff. Those fixes can change behaviour.

- Fighting the formatter. If ruff format reflows your line, accept it; the point is that nobody argues about layout in reviews.

- Treating a clean lint as proof of correctness. ruff does not know what your function should return; tests do.

## Concept 6: mypy, checking types before the code runs

### Why it matters for an FDE

On Day 5 you added type hints, and Python ignores them at runtime. mypy is the tool that reads them and checks that the code agrees, without running it, the way the TypeScript compiler checks a .ts file.

For an FDE this matters most at the boundaries: data from a customer API, an LLM's structured output (Week 6), a CMS entry with optional fields. "This might be None" is the most common crash in integration code, and mypy finds it at your desk instead of in the customer's logs.

### How it works

mypy follows the hints from function to function. Here it finds three bugs that would each crash or corrupt data at runtime:

In [ ]:
# Day 12: three bugs mypy finds without running the code
from dataclasses import dataclass


@dataclass
class Entry:
    uid: str
    title: str
    word_count: int
    author: str | None = None  # "str | None": the value may be missing


def reading_minutes(entry: Entry) -> int:
    return entry.word_count / 200  # BUG 1: "/" always gives a float


def author_initials(entry: Entry) -> str:
    return entry.author[0].upper()  # BUG 2: author can be None


def total_words(entries: list[Entry]) -> int:
    return sum(e.word_count for e in entries)


print(total_words(Entry("blt001", "SSO", "820")))  # BUG 3: two in one

# $ mypy typed.py   (messages shortened to fit)
# -> typed.py:14: error: Incompatible return value type
#        (got "float", expected "int")  [return-value]
# -> typed.py:18: error: Value of type "str | None" is not indexable
#        [index]
# -> typed.py:25: error: Argument 1 to "total_words" has incompatible
#        type "Entry"; expected "list[Entry]"  [arg-type]
# -> typed.py:25: error: Argument 3 to "Entry" has incompatible type
#        "str"; expected "int"  [arg-type]
# -> Found 4 errors in 1 file (checked 1 source file)

Line 25 gets two errors from one call: a str where Entry wants an int, and an Entry where total_words wants a list. Running the file would only show you the first crash it reached, on whatever data happened to trigger it.

To fix "str | None is not indexable", you narrow the type: after if entry.author is None: return "", mypy knows that entry.author is a str for the rest of the function. reveal_type(x) asks mypy which type it sees, which is the fastest way to understand a confusing error.

In [ ]:
# Day 12: fixing mypy errors by narrowing, and what --strict adds
from dataclasses import dataclass
from typing import reveal_type


@dataclass
class Entry:
    uid: str
    word_count: int
    author: str | None = None


def reading_minutes(entry: Entry) -> int:
    return max(1, entry.word_count // 200)  # "//": whole-number division


def author_initials(entry: Entry) -> str:
    if entry.author is None:  # narrowing: mypy follows this check
        return ""
    reveal_type(entry.author)  # asks mypy which type it sees here
    return entry.author[0].upper()  # safe: author is a str on this line


def legacy_to_row(e, default_locale="en-us"):  # no hints at all
    return {"uid": e["uid"], "locale": e.get("locale") or default_locale}


print(author_initials(Entry("blt001", 820, "priya")))  # -> P
print(reading_minutes(Entry("blt002", 150)))  # -> 1  (max keeps 1 min)

# $ mypy fixed.py
# -> fixed.py:20: note: Revealed type is "str"
# -> Success: no issues found in 1 source file   (legacy_to_row skipped!)
# $ mypy --strict fixed.py
# -> fixed.py:24: error: Function is missing a type annotation
# GOTCHA: at runtime reveal_type also prints "Runtime type is 'str'" to
# stderr, so delete it once you have your answer.

Strict mode: by default mypy skips functions that have no hints at all, so a legacy file with zero annotations passes with "no issues" and proves nothing. strict = true in [tool.mypy] makes missing hints an error and turns on a dozen other checks. Use strict for new code such as today's package; on a large legacy codebase, turn it on one module at a time.

Types you will write today: Iterator[Entry] for what a generator yields and Iterable[Row] for anything you can loop over once (both from collections.abc), str | None for optional values, and aliases such as Entry = dict[str, Any] to name a long type once. Any switches checking off for that value, so keep it at the edge (raw JSON) and convert to precise types as early as you can.

py.typed: an empty file inside the package that tells mypy, when it runs in someone else's project, that your package ships type hints. Without it, their mypy treats your package as untyped.

### Real-world FDE example

Your Week 11 agent reads Contentstack entries whose author reference is optional. The demo works because every demo entry has an author. Two days into the pilot, the agent crashes on the first entry without one, in front of the customer's editors. With strict mypy and author: Author | None in your model, the line entry.author.name would have been an error on your laptop. That is why strict typing is worth its small cost on any code that touches customer data.

### Common mistakes

- Silencing errors with # type: ignore or Any instead of fixing them. Each one is a hole a real bug can hide in. If you must, name the code: # type: ignore[arg-type].

- Trusting a clean mypy run on unannotated code. Without hints, or without strict, mypy checks almost nothing.

- Annotating a generator as returning list[...]. A generator returns an Iterator; mypy will tell you, but make sure you understand why (Day 11).

- Writing typing.List and typing.Dict. Since Python 3.9, write list[str] and dict[str, int]; ruff's UP rules flag the old forms.

## Check your understanding (predict first, then run)

Save this as check12.py in your scratch folder. Write a prediction for each numbered part, then run python3 check12.py, then the two mypy commands in part 4. If one surprises you, find the line responsible and explain it in the notebook.

In [ ]:
# Day 12 check: predict each numbered part, then run python3 check12.py
import logging
import sys

# 1 which messages appear, and on which stream?
logging.basicConfig(
    level=logging.WARNING,
    stream=sys.stdout,  # note: stdout this time
    format="%(levelname)s %(message)s",
)
log = logging.getLogger("check")
log.info("one")
log.warning("two")

# 2 does a second basicConfig call change anything?
logging.basicConfig(level=logging.DEBUG)
log.debug("three")

# 3 a child logger with its own level
logging.getLogger("check.child").setLevel(logging.ERROR)
logging.getLogger("check.child").warning("four")
logging.getLogger("check.child").error("five")


# 4 then run: mypy check12.py   and   mypy --strict check12.py
def add_gst(amount):
    return amount * 1.18


def label(count: int) -> str:
    return "entries: " + str(count)


print(add_gst("100"), label(3))

- Part 1: which of "one" and "two" appear, and on stdout or stderr?

- Part 2: does "three" appear? Why or why not?

- Part 3: which of "four" and "five" appear, and why?

- Part 4: what happens when the file runs, what does plain mypy report, and what does mypy --strict report? What does that tell you about running mypy on a legacy codebase?

## Hands-on task (about 2.5 hours): the cms-export package

The scenario: a customer's platform lead says: "Our content migration runs on one script that only works on Priya's laptop. It prints to the console, crashes on some exports, and nobody can install it on the job runner. Make it something any engineer can install with one command, that logs properly into our log platform (we collect stderr), and that our CI can lint and type check."

Before you start: git switch main && git pull, then git switch -c day12. In day12/ you get the customer's script legacy_cms_export.py (it works; read it, do not edit it), the sample export_sample.jsonl (9 lines: one blank, one broken, a draft, an archived entry and one entry with no locale) and the starter project cms-export/. Every function you write has a comment block with its rules and HINT lines, and a body of raise NotImplementedError("TODO"). The two test files are given; do not edit them.

```bash
# day12/ as you get it (git pull on main first)
day12/
    legacy_cms_export.py     # the customer's script: read, do not edit
    export_sample.jsonl      # 9 lines: blank, broken, draft, archived...
    dsa.py                   # today's two DSA problems
    day12_notes.ipynb        # runnable examples from this session
    cms-export/              # YOUR project (Part A makes it one)
        .gitignore           # given: .venv/, dist/, out/ and caches
        src/cms_export/      # __init__, __main__, py.typed given;
                             # cli, reader, transform, writer and
                             # logging_setup are starters with TODOs
        tests/               # 2 given test files + data/ sample

# Part A, starting from the repo root
$ cd day12/cms-export
$ uv init --package          # adds pyproject.toml, README, .python-version
$ uv add --dev pytest ruff mypy
$ uv run pytest -q           # red at first: every TODO raises
# -> 10 failed, 1 passed   (the ExportStats test: it is given)
```

Part A, make it a project (30 min). Inside day12/cms-export/ run uv init --package; it adds pyproject.toml, README.md and .python-version and leaves the starter files alone. Then edit pyproject.toml: a real description, requires-python = ">=3.12", the script line pointing at cms_export.cli:main, and the [tool.ruff], [tool.ruff.lint] and [tool.mypy] tables from Concepts 5 and 6, with strict = true and files = ["src"] under mypy. Check that uv run python -c "import cms_export" prints nothing.

Part B, port the logic with types (60 min). Read the legacy script once, top to bottom, and write down its problems (you should find at least six). Then fill in transform.py, reader.py and writer.py. Port the behaviour, not the bugs: no globals, no print, no bare except, and a header row even when nothing is published. Run uv run pytest tests/test_transform.py after each function.

Part C, logging and the command (40 min). Write level_for and configure_logging, then build_parser, published_rows and main in cli.py. Run the command on the sample and compare with the expected output below.

Part D, quality gates and install (20 min). Make uv run ruff check, uv run ruff format --check and uv run mypy pass. Add at least three tests of your own in tests/test_mine.py: a parametrized test for level_for, one proving that -v logs "skipping blt003: status draft", and one for write_csv with zero rows. Then build the wheel and install it into a fresh venv outside the project.

```bash
# What a correct run looks like (inside day12/cms-export). Each log line
# starts with a time such as 2026-11-05T08:00:01, cut here to fit.
$ uv run cms-export ../export_sample.jsonl out/posts.csv
# -> WARNING cms_export.reader: skipping line 6: Expecting ',' delimiter
# -> WARNING cms_export.transform: blt006 has no locale, using en-us
# -> INFO cms_export.cli: read 7 entries, 5 published, bad lines: [6]
# -> INFO cms_export.cli: wrote 5 rows to out/posts.csv
$ uv run cms-export -v ../export_sample.jsonl out/posts.csv
# -> adds DEBUG cms_export.cli: skipping blt003: status draft
# -> and  DEBUG cms_export.cli: skipping blt007: status archived
$ uv run cms-export -q ../export_sample.jsonl out/posts.csv
# -> only the two WARNING lines
$ uv run cms-export missing.jsonl out.csv; echo $?
# -> ERROR cms_export.cli: export file not found: missing.jsonl
# -> 1
$ uv run python -m cms_export --version
# -> cms-export 0.1.0

# Part D: build, then install into a fresh venv OUTSIDE the project
$ uv build
# -> Successfully built dist/cms_export-0.1.0-py3-none-any.whl
$ uv venv ~/fresh-venv
$ uv pip install --python ~/fresh-venv dist/cms_export-0.1.0-py3-none-any.whl
$ ~/fresh-venv/bin/cms-export --version   # Windows: Scripts, not bin
# -> cms-export 0.1.0
```

Acceptance criteria:

- uv run cms-export ../export_sample.jsonl out/posts.csv writes the four log lines shown to stderr (your timestamps differ), nothing to stdout, and exits with 0. out/posts.csv has the header and 5 rows (blt001, blt002, blt004, blt006, blt008), titles stripped, blt006 with locale en-us.

- -v adds the two DEBUG skipping lines; -q shows only the two WARNING lines; a missing source file logs ERROR export file not found: ... and exits with 1; no arguments prints usage and exits with 2; --version prints cms-export 0.1.0, read from the package metadata.

- uv run python -m cms_export ... behaves exactly like the cms-export command.

- uv run pytest: the 11 given tests plus at least 3 of yours, all green.

- uv run ruff check, uv run ruff format --check and uv run mypy (strict, on src) all pass, with no noqa or type: ignore comments.

- No print( anywhere in src/, no module-level mutable state, and only cli.py calls configure_logging.

- uv build makes a wheel, and after installing it into a fresh venv outside the project, cms-export --version works there.

- pyproject.toml and uv.lock are committed; .venv/, dist/ and out/ are not (the given .gitignore covers them).

- The work is on the branch day12 in at least two commits (project skeleton; port with logging and checks), pushed, with a pull request that has What, Why, How to test and your list of legacy problems.

Hints (no solutions, try first):

- Work in the order the modules import each other: transform first (it imports nothing of yours), then reader and writer, then cli. That is also the dependency direction from Concept 1.

- If uv run cms-export fails with ModuleNotFoundError or "cannot import name main", check the [project.scripts] line, then run uv sync again: commands are created at install time.

- If the CLI tests see no log lines, read the GOTCHA at the end of Concept 4 again, then look at how you set the level.

- If mypy reports an error you do not understand, put reveal_type(...) on the line above, run mypy again, and remove it afterwards.

- The pipeline is lazy: nothing is read until write_csv pulls rows through it. Log the summary after write_csv returns, or every count will be 0.

- Before you finish, search src/ for print( and for basicConfig. Each should appear exactly where the acceptance criteria allow, and nowhere else.

## DSA practice (30 min)

Both problems are in day12/dsa.py with signatures, examples and hints. Both split a string into parts and compare the parts, the pattern behind every version check and log parser.

- Compare versions. compare_versions("1.10.0", "1.9.3") returns 1, ("0.11.32", "0.12.0") returns -1, and ("2.0", "2.0.0") returns 0. Hint: comparing the strings directly gets 1.10 against 1.9 wrong; split on dots, convert each part to int, and treat a missing part as 0.

- Common package prefix. common_package(["cms_export.reader", "cms_export.cli"]) returns "cms_export", and ["a.b.c", "a.b.d", "a.b"] gives "a.b". Hint: split each name on dots, then walk position by position with zip(*parts) and stop at the first position where the parts differ. Whole parts only: "reader" and "readers" share nothing.

## Quiz (3 quick questions)

- A teammate adds logging.basicConfig(level=logging.DEBUG) at the top of reader.py "so the debug lines show". What goes wrong for the customer's application that imports your package, and where should that line live?

- The customer's CI runs pip install -r requirements.txt with no versions pinned, and a deploy broke after a dependency released a new version. What do you change in your project, and which command should their CI run?

- mypy reports Item "None" of "str | None" has no attribute "upper". Explain the error in one sentence, give two ways to fix it, and say which you prefer for data coming from a customer's CMS.

## What you achieved today

You can turn a script into a real Python project: a src-layout package with one job per module, a command created from an entry point, metadata and dependencies in pyproject.toml, and an exact, reproducible environment from uv.lock. You replaced print with logging that a customer's platform can filter by level and module, and you can prove log behaviour with caplog. ruff and mypy now review every change before a person does, and you know what each can and cannot prove. Every project in this course starts this way from now on.

Tomorrow (Day 13): async Python with asyncio, async/await and httpx, used to build a concurrent CMS fetcher that keeps at most 3 requests in flight.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 12 review": your recap answers and check-your-understanding predictions next to the real outputs, your pyproject.toml, the five module files and tests/test_mine.py, the output of uv run pytest -v, uv run ruff check and uv run mypy, the cms-export --version run from the fresh venv, your list of legacy problems, dsa.py, the pull request link and your quiz answers. Say which concept felt least natural; tomorrow's recap starts there.

## Recap answers

*Your answers:*
